# 03 - Exploratory Data Analysis (EDA)
In this notebook, we perform exploratory data analysis on the cleaned and cancelled ride datasets to uncover patterns regarding ride cancellations, vehicle types, time of day, and location.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
%matplotlib inline
sns.set_style('whitegrid')


## 1. Load Datasets


In [ ]:
cleaned_df = pd.read_csv('../data/processed/cleaned_dataset.csv')
cancelled_df = pd.read_csv('../data/processed/cancelled_dataset.csv')

# Create a binary target column for Cancellation (1 = Canceled/Driver Not Found, 0 = Success)
cleaned_df['Is_Cancelled'] = cleaned_df['Booking_Status'].apply(lambda x: 0 if x == 'Success' else 1)

print("Cleaned Data Shape:", cleaned_df.shape)
print("Cancelled Data Shape:", cancelled_df.shape)


## 2. Cancellation Rates vs Raw Counts
Instead of just looking at raw counts, we will calculate the cancellation rate (%) to accurately identify problem areas.


In [ ]:
def plot_cancellation_rate(df, column, title):
    rate_df = df.groupby(column)['Is_Cancelled'].mean().reset_index()
    rate_df['Is_Cancelled'] *= 100 # Convert to percentage
    rate_df = rate_df.sort_values(by='Is_Cancelled', ascending=False)
    
    plt.figure(figsize=(12, 6))
    sns.barplot(data=rate_df, x=column, y='Is_Cancelled', palette='Reds_r')
    plt.title(title)
    plt.ylabel('Cancellation Rate (%)')
    plt.xticks(rotation=45)
    plt.show()

plot_cancellation_rate(cleaned_df, 'Is_Peak_Hour', 'Cancellation Rate by Peak Hour')
plot_cancellation_rate(cleaned_df, 'Vehicle_Type', 'Cancellation Rate by Vehicle Type')
plot_cancellation_rate(cleaned_df, 'Payment_Method', 'Cancellation Rate by Payment Method')
plot_cancellation_rate(cleaned_df, 'Hour', 'Cancellation Rate by Hour of Day')


## 3. Correlation Heatmap
Understanding relationships between numerical variables like distance, fare, rating, and cancellation.


In [ ]:
numerical_cols = ['Ride_Distance', 'Booking_Value', 'Driver_Ratings', 'Customer_Rating', 'Hour', 'Is_Cancelled']
corr_matrix = cleaned_df[numerical_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap of Numerical Variables')
plt.show()


## 4. Day-of-Week Analysis
Analyzing cancellation patterns across different days of the week.


In [ ]:
if 'Day_of_Week' in cleaned_df.columns:
    # Sort days of week correctly
    days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    cleaned_df['Day_of_Week'] = pd.Categorical(cleaned_df['Day_of_Week'], categories=days_order, ordered=True)
    plot_cancellation_rate(cleaned_df, 'Day_of_Week', 'Cancellation Rate by Day of Week')
else:
    print('Day_of_Week column not found in dataset.')


## 5. Driver vs Customer Cancellations by Distance
Comparing ride distance distributions specifically between Driver Cancellations, Customer Cancellations, and Success.


In [ ]:
plt.figure(figsize=(10, 6))
# Filter to relevant statuses to compare Canceled By Driver, Canceled By Customer, and Success
status_filter = cleaned_df[cleaned_df['Booking_Status'].isin(['Success', 'Canceled By Driver', 'Canceled By Customer'])]

sns.boxplot(data=status_filter, x='Booking_Status', y='Ride_Distance', palette='Set2')
plt.title('Ride Distance by Booking Status')
plt.ylabel('Ride Distance (km)')
plt.show()


## 6. Cancellation Reason Breakdown
Exploring the specific reasons for cancellations from the cancelled dataset safely handling N/A and NaN values.


In [ ]:
# Driver Cancellation Reasons
driver_reasons = cancelled_df['Canceled_Rides_by_Driver'].replace('N/A', np.nan).dropna().value_counts()
plt.figure(figsize=(10, 5))
sns.barplot(y=driver_reasons.index, x=driver_reasons.values, palette='Blues_r')
plt.title('Top Reasons for Driver Cancellations')
plt.xlabel('Count')
plt.show()

# Customer Cancellation Reasons
customer_reasons = cancelled_df['Canceled_Rides_by_Customer'].replace('N/A', np.nan).dropna().value_counts()
plt.figure(figsize=(10, 5))
sns.barplot(y=customer_reasons.index, x=customer_reasons.values, palette='Oranges_r')
plt.title('Top Reasons for Customer Cancellations')
plt.xlabel('Count')
plt.show()
